# **Earnings Call Sentiment & Return Predictability**

Investigating whether linguistic signals from earnings call transcripts predict future stock returns.

**Status:** In development — core pipeline implemented, backtesting in progress.

---

### Pipeline
1. Transcripts
2. Speaker-level parsing
3. FinBERT sentiment
4. Event-level features
5. CRSP alignment
6. Forward return construction
7. Long/short backtest

### Key Features
- `exec_sentiment` — average executive tone
- `sentiment_gap` — exec / analyst disagreement  
- `qna_pressure` — analyst tone in Q&A section

### Data
- Earnings call transcripts via LSEG/Refinitiv (academic license)
- CRSP daily returns and index data via WRDS

*For research purposes only. Raw data not redistributed.*

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F


In [ ]:
russell_df = pd.read_excel("russell_universe.xlsx")
russell_df.head()

,Identifier (RIC),Company Name,Exchange Ticker,Market Cap\n(Σ=None),GICS Sector Name,Exchange Name
0,Totals (1004),NaN,NaN,NaN,NaN,NaN
1,.RUI,Russell Investments,NaN,NaN,NaN,NaN
2,DT.N,Dynatrace Inc,DT,1.022703e+10,Information Technology,New York Stock Exchange
3,PAYC.N,Paycom Software Inc,PAYC,5.975516e+09,Industrials,New York Stock Exchange
4,PR.N,Permian Resources Corp,PR,1.757226e+10,Energy,New York Stock Exchange


In [ ]:
russell_df.rename(columns={'Identifier (RIC)': 'RIC','Company Name':'company_name', 'Exchange Ticker':'exchange_ticker','Market Cap\n(Σ=None)':'market_cap','GICS Sector Name': 'sector','Exchange Name':'exchange'}, inplace=True)
##Data cleaning
russell_df['RIC'] = russell_df['RIC'].str.strip()
russell_df = russell_df[russell_df['RIC'].str.match(r"^[A-Z0-9]+\.[A-Z]$",na=False)].copy()
russell_df['RIC'].nunique()

In [ ]:
def get_transcripts(folder):
    from pathlib import Path
    import pandas as pd

    rows = []

    for file in Path(folder).glob("*.txt"):
        parts = file.stem.split("-")

        # must have at least date + ticker
        if len(parts) < 4:
            continue

        date_str = "-".join(parts[0:3])
        ticker = parts[3]

        try:
            date = pd.to_datetime(date_str)
        except:
            continue

        rows.append({
            "ticker": ticker,
            "date": date,
            "file_path": str(file)
        })

    return pd.DataFrame(rows)

In [ ]:
def extract_date(text):
    match = re.search(r'(\w+ \d{1,2}, \d{4})', text)
    if match:
        return datetime.strptime(match.group(1), '%B %d, %Y').strftime('%Y-%m-%d')
    return None

In [ ]:
def clean_transcript(text):

    text = re.split(r'\n(?:Definitions|Disclaimer|Copyright)\s*\n', text, flags=re.IGNORECASE)[0]

    date = extract_date(text)

    blocks = re.split(r'-{20,}', text)

    speaker_turns = []
    current_speaker_line = None
    current_text = None
    in_qna = False

    for block in blocks:
        block = block.strip()
        if not block:
            continue

        if any(x in block for x in [
            "StreetEvents", "EDITED VERSION", "UTC",
            "Corporate Participants", "Conference Call Participants",
            "==============="
        ]) or block.lower() == "presentation":
            continue

        if re.search(r'question.and.answer|q&a|questions and answers', block, re.IGNORECASE):
            in_qna = True
            continue

        speaker_match = re.match(r'^(.+?)\s+\[(\d+)\]\s*$', block)

        if speaker_match:
            if current_speaker_line and current_text:
                name, role = classify_role(current_speaker_line)
                speaker_turns.append({
                    'speaker': name,
                    'role': role,
                    'date': date,
                    'section': 'qna' if in_qna else 'presentation',
                    'text': current_text
                })
            current_speaker_line = speaker_match.group(1).strip()
            current_text = None
        else:
            if current_text:
                current_text += "\n" + block
            else:
                current_text = block

    if current_speaker_line and current_text:
        name, role = classify_role(current_speaker_line)
        speaker_turns.append({
            'speaker': name,
            'role': role,
            'date': date,
            'section': 'qna' if in_qna else 'presentation',
            'text': current_text
        })

    return speaker_turns

In [ ]:
##load transcripts
transcript_df = get_transcripts("/content/data/transcripts")

In [ ]:
print(transcript_df["ticker"].nunique())
print(transcript_df["date"].min())
print(transcript_df["date"].max())
print(transcript_df.isna().sum())

7
2022-11-03 00:00:00
2026-04-23 00:00:00
ticker       0
date         0
file_path    0
dtype: int64


In [ ]:
def load_text(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

transcript_df["text"] = transcript_df["file_path"].apply(load_text)

In [ ]:
all_rows = []

for _, row in transcript_df.iterrows():
      turns = clean_transcript(row["text"])

      for t in turns:
          t["ticker"] = row["ticker"]
          all_rows.append(t)

features_df = pd.DataFrame(all_rows)

In [ ]:
cleaned_records = features_df.copy()

# **FinBERT (Baseline)**


FinBERT is a pre-trained NLP model developed by researchers at the University of Hong Kong, built on top of Google's BERT (Bidirectional Encoder Representations from Transformers) architecture. It was specifically fine-tuned for financial text, making it better suited for finance-related NLP tasks than general purpose BERT.

**Training Data:**

Pre-trained on a large financial corpus including Reuters news, Bloomberg articles, and 10-K/10-Q SEC filings
Fine-tuned for sentiment analysis on the Financial PhraseBank dataset, which consists of roughly 5000 sentences from financial news manually labeled as positive, negative, or neutral by financial experts


**Limitations:**

Only outputs 3 sentiment classes
Trained mostly on news/filings, so performance may vary on earnings call transcripts specifically
Doesn't understand numerical context (e.g. it reads "revenue grew 2%" and "revenue grew 200%" the same way)

In [ ]:
model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
from torch.utils.data import DataLoader

def get_sentiment_batch(texts, batch_size=32):
    all_scores = []
    all_confidences = []

    # process in batches
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True  # pad to same length within batch
        )

        with torch.no_grad():
            outputs = model(**inputs)

        probs = F.softmax(outputs.logits, dim=1)

        for prob in probs:
            neg, neu, pos = prob.tolist()
            all_scores.append(pos - neg)
            all_confidences.append(max(prob.tolist()))

    return all_scores, all_confidences

* High confidence (0.85–0.99):
text is strongly negative or positive
* Medium confidence (0.5–0.7):
mixed tone, cautious language
* Low confidence (0.33–0.45):
neutral or vague language

In [ ]:
records_df = pd.DataFrame(cleaned_records)
##drop operator for modeling purposes
records_df = records_df[records_df['role'] != 'Operator']

In [ ]:
texts = records_df['text'].tolist()
scores, confidences = get_sentiment_batch(texts, batch_size=32)

In [ ]:
records_df['sentiment'] = scores
records_df['confidence'] = confidences

In [ ]:
print(records_df["sentiment"].isna().sum())
records_df["text"].str.len().describe()

0


,text
count,808.000000
mean,830.128713
std,1183.259359
min,7.000000
25%,211.750000
50%,458.500000
75%,941.000000
max,11001.000000


Sentiment Analysis

In [ ]:
exec_df = records_df[records_df["role"] == "Executive"]
analyst_df = records_df[records_df["role"] == "Analyst"]

In [ ]:
##executive sentiment
exec_sent = exec_df.groupby(["ticker", "date"])["sentiment"].mean().reset_index(name='exec_sent')

In [ ]:
##analyst sentiment
analyst_sent = analyst_df.groupby(["ticker", "date"])["sentiment"].mean().reset_index(name='analyst_sent')

* positive gap: management more optimistic than market
* negative gap: management more pessimistic than analysts

In [ ]:
baseline_features = pd.merge(exec_sent, analyst_sent, on=["ticker", "date"], how="inner")
##sentiment gap
baseline_features['sentiment_gap'] = baseline_features['exec_sent']-baseline_features['analyst_sent']

* high std: inconsistent messaging / uncertainty
* low std: stable narrative

In [ ]:
dispersion = exec_df.groupby(["ticker", "date"])["sentiment"].std().reset_index(name='dispersion')

High Q&A pressure usually means:
analysts are asking tougher questions
more negative or probing language
uncertainty about outlook, margins, guidance
management is being pushed to clarify or defend

Often linked to:

higher perceived risk
greater uncertainty
post-earnings volatility
Low Q&A pressure usually means:
routine questions
confirmation of strong performance
fewer challenges to management narrative
smoother information flow

Often linked to:

stable outlook
lower uncertainty
less disagreement

In [ ]:
qa_pressure = analyst_df[analyst_df["section"] == "qna"].groupby(['ticker','date'])["sentiment"].mean().reset_index(name='qa_pressure')

In [ ]:
baseline_features = (
    baseline_features
    .merge(dispersion, on=['ticker','date'], how='left')
    .merge(qa_pressure, on=['ticker','date'], how='left')
)

In [ ]:
high_threshold = baseline_features["qa_pressure"].quantile(0.75)
very_high_threshold = baseline_features["qa_pressure"].quantile(0.90)
baseline_features["qa_high"] = (baseline_features["qa_pressure"] > high_threshold).astype(int)
baseline_features["qa_very_high"] = (baseline_features["qa_pressure"] > very_high_threshold).astype(int)

**CRSP**

Data is sourced directly from WRDS CRSP via authenticated Python API access, ensuring reproducibility and eliminating manual preprocessing bias.

In [ ]:
pip install wrds

In [ ]:
def load_crsp(db):
    return db.raw_sql("""
        SELECT d.permno, d.dlycaldt, d.dlyret, d.dlyclose, d.dlyvol, d.shrout
        FROM crsp.dsf_v2 d
        JOIN crsp.msenames n ON d.permno = n.permno
        WHERE d.dlycaldt BETWEEN '2010-01-01' AND '2026-04-25'
        AND n.shrcd IN (10,11)
        AND n.exchcd IN (1,2,3)
        AND d.dlycaldt BETWEEN n.namedt AND n.nameendt
    """)

crsp = load_crsp(db)
crsp.to_parquet("crsp_dsf.parquet")

In [ ]:
def load_name_mapping(db):
    return db.raw_sql("""
        SELECT permno, ticker, namedt, nameendt
        FROM crsp.msenames
    """)

names = load_name_mapping(db)
names.to_parquet("crsp_names.parquet")

In [ ]:
def load_market(db):
    return db.raw_sql("""
        SELECT date, vwretd
        FROM crsp.dsi
        WHERE date BETWEEN '2010-01-01' AND '2026-04-25'
    """)
market = load_market(db)
market.to_parquet("crsp_dsi.parquet")

In [ ]:
crsp = pd.read_parquet("crsp_dsf.parquet")
names = pd.read_parquet("crsp_names.parquet")
market = pd.read_parquet("crsp_dsi.parquet")

In [ ]:
crsp["dlycaldt"] = pd.to_datetime(crsp["dlycaldt"])
market["date"] = pd.to_datetime(market["date"])

In [ ]:
crsp = crsp.merge(
    market,
    left_on="dlycaldt",
    right_on="date"
)

In [ ]:
baseline_features = baseline_features.merge(names, on="ticker")
baseline_features = baseline_features[
    (baseline_features["date"] >= baseline_features["namedt"]) & ##ensure ticker is matched in time
    (baseline_features["date"] <= baseline_features["nameendt"])
]

In [ ]:
##adjust for market return
crsp['abnormal_ret'] = crsp['dlyret']-crsp['vwretd']

In [ ]:
crsp = crsp.sort_values(by=['permno', 'dlycaldt'])

# return at t+1
crsp['ret_1d'] = crsp.groupby('permno')['abnormal_ret'].shift(-1)

# return at t+5
crsp['log_ret'] = np.log(1 + crsp['abnormal_ret'])

crsp['ret_5d'] = (
    crsp.groupby('permno')['log_ret']
    .transform(lambda x: x.shift(-5).rolling(5).sum())
)
crsp['ret_5d'] = np.exp(crsp['ret_5d']) - 1

# drop helper column
crsp.drop(columns=['log_ret'], inplace=True)

In [ ]:
final = baseline_features.merge(
    crsp,
    left_on=["permno", "date"],
    right_on=["permno", "dlycaldt"]
)

In [ ]:
y = final[['ret_5d']]
X = final[['sentiment_gap','qa_pressure','dispersion','exec_sent']]

In [ ]:
##check for monotonicity by splitting sentiment gap into 4 quartiles
final['gap_q'] = pd.qcut(final['sentiment_gap'],4, labels=False)
top = final[final['gap_q'] == 3]['ret_5d'].mean() ##top quartile
bottom = final[final['gap_q'] == 0]['ret_5d'].mean() ##bottom quartile

spread = top - bottom
print("Top Quartile:", top)
print("Bottom Quartile:", bottom)
print("Spread:", spread)